#
# Predictions from the Model are verified in the Neural Recordings
#

### Exploring Behavioral and Neural Predictions from Modelling the Decision Manifold Geometry

This notebook explores the **behavioral and neural predictions** that arise from a retinotopic model accounting for the structure of the decision manifold uncovered in LIP neural population activity.

We focus on two key analyses:

1. **Single-Neuron Choice Selectivity and Reaction Time**  
   We examine how a neuron’s choice selectivity varies across the manifold, particularly as a function of **reaction time**. This allows us to test whether neurons show enhanced selectivity under specific dynamical regimes, and whether fast or slow decisions engage different subpopulations of LIP neurons.

2. **Saccade Endpoint Predictions**  
   We then test the hypothesis that **shifts in neural population activity** translate into measurable differences in behavior. Specifically, we use **pupil tracking data** to assess whether **saccade endpoints** differ systematically between **fast** and **slow** trials. This analysis reveals that in most sessions, the distribution of saccade landing positions varies with reaction time—and in some cases, slow saccades are biased toward the midpoint between the targets.

Together, these findings provide converging evidence that LIP population activity is tightly coupled to both decision formation and motor execution—and that manifold coordinates such as reaction time offer a useful lens for interpreting both.

In [ ]:
import os
import sys

# Set the root directory

# Automatically find the project root (directory containing 'src' or 'data')
def find_project_root(marker_dirs=("src", "data","notebooks")):
    path = os.getcwd()
    while path != "/" and not all(os.path.exists(os.path.join(path, d)) for d in marker_dirs):
        path = os.path.dirname(path)
    return path

PROJECT_ROOT = find_project_root()

# Set up Python import path and working directory
sys.path.append(os.path.join(PROJECT_ROOT, "src"))
os.chdir(PROJECT_ROOT)

print("Project root:", PROJECT_ROOT)




In [ ]:
import pickle
import gzip
from src.io_utils import download_session
from src.io_utils import load_dataframe_with_metadata
from scipy.ndimage import gaussian_filter1d
import numpy as np
from src.geometry import compute_geometry_measures
from src.geometry import local_average
import os


# Download the results to avoid computing it
# Comment this line if you want to compute the results from scratch (slow)
path = download_session('population_local_average')


# Crop time and align trials LFADS
Cut = 20     # time steps (10ms bins) after dotsOn to trim start
CutEnd = -6  # time steps before saccadeDetected to trim end
sigma = 5    # smoothing kernel width

# Check if the results file already exists
pops_file = 'data/population_local_average.pkl.gz'

if os.path.exists(pops_file):
    print(f"Loading population local trial-averages from {pops_file}")
    with gzip.open(pops_file, 'rb') as f:
        pops = pickle.load(f)
else:
    print("Computing population local trial-averages ...\n")
    pops = []

    # Iterate over all sessions
    for session in [f'S{i}' for i in range(1, 9)]:
        path = download_session(session)
        df = load_dataframe_with_metadata(session)
       # print(df.attrs[''])

        # Extra smoothing LFADS trajectories
        for trial in df.index:
            df.at[trial, 'LFADS'] = gaussian_filter1d(df.at[trial, 'LFADS'], sigma=sigma, axis=1)

        # Compute reference starting points for alignment
        start_vectors = np.stack([df.at[trial, 'LFADS'][:, 0] for trial in df.index], axis=1)
        mean_start = np.mean(start_vectors, axis=1, keepdims=True)

        # Align and cut LFADS + store back
        for trial in df.index:
            traj = df.at[trial, 'LFADS']
            aligned = traj[:, Cut:CutEnd] - traj[:, [0]] + mean_start
            df.at[trial, 'LFADS'] = aligned

        # Compute geometry measures
        compute_geometry_measures(df, column='LFADS', timeres=10, arc_res=101)

        # Compute local averages for each choice
        for choice in [0, 1]:
            pop_locav, nba_locav = local_average(
                df[df['choice'] == choice],
                column='LFADS-Arc',
                behavioral_axis='RT',
                frac=0.5,
                ba_res=101,
                method='lowess',
                remove_outliers=0.01
            )
            pops.append({
                'session': session,
                'choice': choice,
                'new_behavioral_axis': nba_locav,
                'pop_locav': pop_locav
            })

    # Save the results to a pickle gzip file
    with gzip.open(pops_file, 'wb') as f:
        pickle.dump(pops, f)


import pandas as pd
pops = pd.DataFrame(pops)


# Optional: apply smoothing across arc-length (axis=1) after loading
extra_arc_smoothing = True
arc_smoothing_sigma = 10  # in arc-length points

if extra_arc_smoothing:
    from scipy.ndimage import gaussian_filter1d
    for i in range(len(pops)):
        pops.at[i, 'pop_locav'] = gaussian_filter1d(pops.at[i, 'pop_locav'], sigma=arc_smoothing_sigma, axis=1)

##
## Neural Prediction: Single-Neuron Choice Selectivity

The `choice_selectivity_binary` function computes the choice selectivity for each neuron and behavioral axis point (RT) in a running window of arc length. It takes two input arrays, `choice_0` and `choice_1`, which represent the population local trial-averages for two different choices, and an optional parameter `arc_window` to define the window size as a fraction of the arc length.

The function calculates the choice selectivity as `(1 - Pearson correlation) / 2` between the two choices for each neuron and RT point within the running window. It pads the arc boundaries with NaN values to handle edge cases. The output is an array of the same shape as the input arrays, containing the computed selectivity values.

In [ ]:
from src.properties import choice_selectivity_binary

pop_0 = pops[(pops['session'] == 'S6') & (pops['choice'] == 0)]['pop_locav'].values[0]
pop_1 = pops[(pops['session'] == 'S6') & (pops['choice'] == 1)]['pop_locav'].values[0]

cs_6 = choice_selectivity_binary(pop_0, pop_1, arc_window=0.2)


The following analysis illustrates that on the average, single neurons have complex patterns of choice selectivity within the decision manifold. This is the result of non-trivial firing patterns that depend on the manifold coordinates. Such single-neuron tuning curves reflect the projection of the decision manifold-- exhibiting non-trivial curvature --into a constant direction in state space, that corresponding to a single cell.

In [ ]:
from src.plot_utils import plot_local_average

example_cell_index = 27

plot_local_average(pop_0[example_cell_index], var_name='Firing Rate (Hz)', title = 'Activity of the example cell for the contra-lateral choice').show()
plot_local_average(pop_1[example_cell_index], var_name='Firing Rate (Hz)', title = 'Activity of the example cell for the ipsi-lateral choice').show()
plot_local_average(cs_6[example_cell_index],heatmap_scale='earth_r',smooth_sigma=None,var_name='Choice Selectivity', title = 'Choice Selectivity in running Arc-Length Window').show()

In [ ]:
from src.properties import choice_selectivity_binary
import pandas as pd
import numpy as np


cs_data = []

sessions = pops['session'].unique()

for session in sessions:
    pop0 = pops[(pops['session'] == session) & (pops['choice'] == 0)]['pop_locav'].values[0]
    pop1 = pops[(pops['session'] == session) & (pops['choice'] == 1)]['pop_locav'].values[0]

    # Compute choice selectivity
    cs = choice_selectivity_binary(pop0, pop1, arc_window=0.2)

    # Compute per-choice ranges
    r0 = np.max(pop0, axis=(1, 2)) - np.min(pop0, axis=(1, 2))
    r1 = np.max(pop1, axis=(1, 2)) - np.min(pop1, axis=(1, 2))

    # The minimum range across both choices
    #cell_range = np.minimum(r0, r1)
    cell_range = np.maximum(r0, r1)

    cs_data.append({
        'session': session,
        'choice_selectivity': cs,
        'cell_range': cell_range
    })

# Convert to DataFrame
cs_df = pd.DataFrame(cs_data)

###
### Population-Level Analysis: Choice Selectivity Raster (Arc Slice)

This heatmap visualizes the population **choice selectivity** at a specific point along the **arc-length** axis for a given session. 

Selectivity is computed over an arc-length window (e.g., 20% of arc range), whose center is the chosen arc slice value.

Only neurons with sufficient dynamic range are shown (minimum response difference across arc × RT for both choices). Neurons are sorted by the difference between their mean bottom 10% reaction time and mean top 10% reaction time.

This view highlights how selective each neuron is to choice at a particular moment along the arc, and how this selectivity evolves across reaction time.

In [ ]:
from src.plot_utils import plot_pop_selectivity_arcslice

plot_pop_selectivity_arcslice(cs_df, session='S6', arc=0.9, cell_range_threshold=0.99)

###
### Population-Level Analysis: Choice Selectivity Across Reaction Time Extremes

This analysis explores how a neuron’s choice selectivity changes depending on reaction time. For each neuron, we compute the average selectivity for trials with the fastest and slowest reaction times—specifically, those falling within the top and bottom percentiles of normalized RT.

This provides insight into how task demands—as reflected by reaction time—affect a neuron’s discriminatory power.

The resulting scatter plot displays all neurons across sessions that exceed a threshold for dynamic range (i.e., their activity varies meaningfully across conditions). The `percentile` parameter determines what fraction of RT extremes are averaged to compute “fast” and “slow” selectivity.

The interactive plot also allows you to hover over individual neurons to view their population activity profiles across arc-length for each choice and RT condition.

In [ ]:


from src.plot_utils import plot_pop_selectivity_scatter

# open_interactive=True will send you to another browser tab for full HTML interactivity
plot_pop_selectivity_scatter(cs_df, pops, arc=0.9, percentile=1, cell_range_threshold=0.99, open_interactive=False)

###
### Retinotopic Organization of Choice Selectivity

In this section, we investigate how the spatial location of a neuron's response field in the visual field relates to its choice selectivity as expressed along the neural decision manifold.

To do this, we first estimate each neuron's retinotopic response field from a passive, visually guided task where decision-making is not required. This provides a spatial map of where each neuron is most responsive in visual space. We then use these response fields to compute a population-weighted average of choice selectivity, effectively projecting manifold selectivity back into retinotopic coordinates.

This analysis reveals a striking spatial organization: neurons that are more selective during fast reaction time trials tend to have response fields positioned further away from the targets, while neurons that exhibit stronger selectivity during slower trials tend to cluster between the target locations. This suggests that different phases or demands of the decision process are associated with distinct retinotopic subpopulations of neurons.


In [ ]:
# Download and load the S6 visually guided task data
# Only session S6 has enough visual field sampling and distributed neural activity in the visual field for this analysis


from src.io_utils import download_session

path = download_session('S6_visually_guided_task')

import gzip
import pickle

with gzip.open(path, 'rb') as f:
    df_vgt = pickle.load(f)



We compute neural response fields by measuring spiking activity during a time window aligned to the onset of the sample target on the screen. To determine the optimal time window for capturing visually driven responses, we systematically vary two parameters: the **center** of the time window (referred to as the `shift`) and the **half-width** of the window (referred to as the `window`). We explore a fine-grained range of values for both parameters betwen 50ms and 200ms (only for combinations starting after target onset).

For each parameter combination, we calculate response fields for all neurons and evaluate how spatially organized they are using **Moran’s I**, a statistical measure of spatial clustering. To avoid being influenced by outliers or noisy neurons, we focus on the top 80% of neurons with the highest Moran’s I scores and select the shift and window size that maximize the median Moran’s I across this group.

This process allows us to identify the temporal window during which neurons exhibit the most spatially coherent retinotopic response fields.

In [ ]:


from src.properties import get_response_fields

ds=get_response_fields(df_vgt, mode='forward',sigma=3.0,shift=0.15,window=.1,NSide=51)
ds

In [ ]:


from src.plot_utils import plot_response_fields

# MI: Moran's I
# SI: Spatial Information
plot_response_fields(ds,cells='TinC',sort='MoransI')

Finally, we compute the difference in choice selectivity between fast and slow reaction times for each neuron. This highlights how strongly a neuron's activity discriminates between choices depending on decision speed.

To visualize where in the visual field these selective signals arise, we weight each neuron's contribution by its response field. This creates a **retinotopic map** of the population-level choice selectivity difference between fast and slow trials. The resulting map reveals how neural selectivity is organized not just in state space, but also across visual space.

To reduce noisy contributions and enhance interpretability, we restrict the analysis to neurons that contribute most strongly to activity changes in the decision manifold. Specifically, we apply a higher dynamic range threshold to include only neurons with substantial modulation across conditions. In addition, each neuron's response field is normalized and sharpened (`heat` parameter) to emphasize its most responsive regions. This ensures that the resulting weighted maps reflect meaningful, localized structure rather than diffuse or noisy activity.

Importantly, the spatial pattern of choice selectivity appears more robust around the contralateral target (located in the lower-left visual field). This is expected, as recordings were obtained exclusively from the hemisphere contralateral to that target, leading to stronger and denser representation in that region. In contrast, interpretation of patterns near the ipsilateral target must be done cautiously due to sparser sampling and limited visibility into that side of the visual field. A more balanced assessment of visual field organization would require bilateral recordings and improved coverage across both hemifields.

In [ ]:

from src.plot_utils import plot_visual_field_choice_selectivity_difference

cs_df_s6 = cs_df[cs_df['session'] == 'S6']
th_csrange=np.percentile(cs_df_s6['cell_range'].iloc[0], 50)
mask = cs_df_s6['cell_range'].iloc[0] > th_csrange

plot_visual_field_choice_selectivity_difference(cs_df_session=cs_df_s6, ds=ds, arc=0.9, percentile=1, cell_range_threshold=0.0, heat=15, mask =mask)

##
## Behavioural Prediction: Saccade Endpoints

The model makes a key behavioral prediction: it suggests that the **bump of neural activity** in the retinotopic map of LIP systematically shifts depending on the coordinates of the decision manifold—particularly, on **reaction time**. 

If LIP participates in a distributed system controlling saccadic eye movements, then such shifts in the neural activity may be reflected in behavior—specifically, in the **location of eye saccades**.

To test this hypothesis, we analyze recordings of **pupil position during decision trials**. By examining how saccade endpoints vary as a function of reaction time, we evaluate whether the structure observed in the neural state space is mirrored in the animal's behavioral outputs.

We find that in **most sessions**, the **2D distribution of saccade endpoint locations** differs significantly between the **fastest** and **slowest** trials. In some sessions, the saccades for slower reaction times are **shifted toward the center between the targets**, suggesting complex interactions within the retinotopic organization of LIP in the underlying neural dynamics.

These results support the idea that neural population dynamics modulates not only reaction time and error rates but also might affect motor outputs, and perhaps action accuracy and precision. However, additional modeling approaches and extensions —tailored to each session— might be required to fully characterize these effects and understand the diversity of patterns observed across sessions.

In [ ]:

from src.properties import get_saccade_endpoints

# We make a distinction between targets on the screen and targets in retinotopic (eye) space.
# We attempt to get the target locations in retinotopic space from the visually guided task. 
# However only session S6 mapped the choice targets with enough trials to get a good estimate of the retinotopic target locations.
# Otherwise we use the centroids of saccade locations per choice as anchor points to get the projections to the target line. 
# Interpret this with caution, since the spatial relationship between targets and saccade locations is not trivial (according to the model).

df = get_saccade_endpoints(outlier_percentile=1,filter_correct_trials=False)
df.head()

In [ ]:

from src.plot_utils import plot_saccade_retinotopy
from src.properties import coordinate_transformation


# Centroids in white. Targets in eye coordinates in red computed from the visually guided task when available (only for S6)

# To plot in original coordinates
# plot_saccade_retinotopy(df, session='S6')

# Shifts, rotates, and scales eye space using centroids (or targets when available, e.g. S6) as anchor points
df_line = coordinate_transformation(df,use_targets=False)

plot_saccade_retinotopy(df_line, session='S6')

In [ ]:

from src.plot_utils import plot_retinotopic_RT_map, plot_retinotopic_2d_hist

plot_retinotopic_2d_hist(df_line, bin_size=0.025,percentile=10,  session='S6')
plot_retinotopic_RT_map(df_line, bin_size=0.025, min_trials=3, session='S6')

In [ ]:

from src.plot_utils import plot_retinotopic_1d_hist

plot_retinotopic_1d_hist(df, bin_size=0.025,percentile=10, session='S6', use_targets=False)

###
### Interpreting Saccade Endpoint Shift Statistics

The function `compute_saccade_shift_statistics` quantifies how **saccade endpoints** differ between **fast** and **slow** trials in a decision-making task.

It computes the following:

- **Percentile-Based Trial Splitting**  
  Trials are divided into **fast** and **slow** groups based on the specified `percentile` of **reaction times** (e.g., bottom and top 10%). This allows a direct comparison of eye movement behavior at the extremes of the decision-time distribution.

- **Coordinate Transformation**  
  Saccade endpoints are aligned to a coordinate system defined by the **line joining the two centroids or targets** (`use_targets=True`). In the latter case, the endpoints are projected onto the axis connecting the two targets (and also the orthogonal axis), providing a more behaviorally meaningful reference frame.

- **Wasserstein Distance (2D Shift)**  
  The **2D Wasserstein distance** (specifically, the Sinkhorn approximation) quantifies how much the 2D retinotopic **distribution of saccade endpoints** shifts between fast and slow trials. A higher value means greater dissimilarity in spatial distributions. To assess significance, the function runs a **shuffle test** (`n_shuffles` iterations), randomly permuting trial labels to generate a null distribution of Wasserstein distances. This allows computing a **p-value** for the observed shift, reported relative to `p_lim`.

- **On-Target-Line Shift (1D Projection)**  
  The function also compares the **projected 1D distribution** of saccade endpoints along the line between the two targets. It uses a **one-sided Kolmogorov–Smirnov (KS) test** to determine whether slow saccades are significantly **shifted toward** the midpoint between the targets compared to fast saccades. This helps assess whether slow trials land in more intermediate or dispersed spatial locations. The Off-Target-Line shift is asimilar measure using the two-sided KS test to determine whether slow and fast saccade distributions are signifiacantly different in the orthogonal direction.

  

---

### How to Interpret the Results

- A **significant 2D Wasserstein distance** (p < 0.05) indicates that saccade endpoint locations for fast and slow trials come from **statistically different spatial distributions**.
  
- A **significant one-sided KS test** suggests a directional shift—e.g., slow saccades **landing more centrally** between the targets or systematically biased toward or away from a given target.

- Together, these results help test the model prediction that neural dynamics (as reflected in reaction time) shape not only decision outcomes but also **motor outputs** like saccade landing positions.

In [ ]:

from src.properties import compute_saccade_shift_statistics

compute_saccade_shift_statistics(df, percentile=10, p_lim=0.05, use_targets=True, n_shuffles=100)
